In [1]:
print('Hello, Cloudian 💙 Cloud') 

Hello, Cloudian 💙 Cloud


In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
gemini_api_key = secrets.get_secret("GEMINI_API_TOKEN")
print('HF: ' , hf_token) 
print('Gemini' , gemini_api_key)

- huggingface_hub: Dowload model tu hugging face
- transformters: Load Qwen-2.5VL, Qwen-2.5-Instruct
- diffusers: Load Flux2-Klein-4B
- accelerate: Run model on GPU
- pillow: Read images and solve it
- opencv-python: resize, crop images
- pandas: Read JSON and response result table
- numpy:
- requests: dowload image from url in JSON
- matplotlib: Render images in coordinates 

In [3]:
# cell 1 - Install Dependecies 
!pip -q install \
    huggingface_hub \
    transformers \
    diffusers \
    accelerate \
    safetensors \
    pillow \
    opencv-python \
    pandas \
    numpy \
    matplotlib \
    google-genai
print('💙' * 18)
print("All required packages have been installed. Let's move to the next")
print('💙' * 10)

💙💙💙💙💙💙💙💙💙💙💙💙💙💙💙💙💙💙
All required packages have been installed. Let's move to the next
💙💙💙💙💙💙💙💙💙💙


In [4]:
!pip list

Package                                  Version
---------------------------------------- -------------------
a2a-sdk                                  0.3.26
absl-py                                  1.4.0
accelerate                               1.13.0
access                                   1.1.10.post3
affine                                   2.4.0
aiofiles                                 22.1.0
aiohappyeyeballs                         2.6.1
aiohttp                                  3.13.5
aiosignal                                1.4.0
aiosqlite                                0.22.1
alabaster                                1.0.0
albucore                                 0.0.24
albumentations                           2.0.8
ale-py                                   0.11.2
alembic                                  1.18.4
altair                                   5.5.0
annotated-doc                            0.0.4
annotated-types                          0.7.0
antlr4-python3-runtime       

In [4]:
# ==========================================================
# Cell 2 - Import Libraries
# ==========================================================

import os
import gc
import json
import random
import warnings

from io import BytesIO

import torch
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

from PIL import Image

from google import genai

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)
warnings.filterwarnings("ignore")
# ==========================================================
# Environment
# ==========================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

print("=" * 60)
print(f"Device       : {device}")

if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version : {torch.version.cuda}")
    print(
        f"GPU Memory   : "
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )
else:
    print("Running on CPU")

print("=" * 60)

Device       : cuda
GPU          : Tesla T4
CUDA Version : 12.8
GPU Memory   : 14.56 GB


In [ ]:
# ==========================================================
# Cell 3 - Project Configuration
# ==========================================================

import os
import random
import numpy as np
import torch

# ----------------------------------------------------------
# API Keys
# ----------------------------------------------------------

HF_TOKEN = hf_token
GEMINI_API_KEY = gemini_api_key

# ----------------------------------------------------------
# Paths
# ----------------------------------------------------------

JSON_PATH = "/kaggle/input/datasets/nguynkhanktpm2024/incidents1m/eccv_train.json"

OUTPUT_DIR = "/kaggle/working/Rescue_Image_Generation"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True,
)

# ----------------------------------------------------------
# Models
# ----------------------------------------------------------

GENERAL_MODEL = "google/gemma-3-4b-it" # Lua chon cac model co kha nang nhan hinh anh dau vao duoc 
# Nang cap: google/gemma-4-12B

FLUX_REPO = "black-forest-labs/FLUX.2-klein-4B"
# black-forest-labs/FLUX.2-dev
FLUX_MODEL = FLUX_REPO
# ----------------------------------------------------------
# Generation Parameters
# ----------------------------------------------------------
IMAGE_SIZE = 1024
LIMIT_IMAGES = 5
REQUEST_TIMEOUT = 30
DOWNLOAD_RETRIES = 3
MAX_NEW_TOKENS = 128
NUM_INFERENCE_STEPS = 6 # 15
GUIDANCE_SCALE = 1.0 # 3.5 
BASE_SEED = 50
# ----------------------------------------------------------
# HTTP Headers
# ----------------------------------------------------------

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/137.0 Safari/537.36"
    )
}
# ----------------------------------------------------------
# Random Seed
# ----------------------------------------------------------

random.seed(BASE_SEED)
np.random.seed(BASE_SEED)
torch.manual_seed(BASE_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(BASE_SEED)

# ----------------------------------------------------------
# Summary
# ----------------------------------------------------------

print("=" * 60)
print("Configuration Loaded Successfully")
print("=" * 60)

print(f"Vision Model : {VISION_MODEL}")
print(f"LLM Model    : {LLM_MODEL}")
print(f"FLUX Model   : {FLUX_MODEL}")
print(f"Output Dir   : {OUTPUT_DIR}")

In [19]:
# Cell 4 - Load Dataset

with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Total samples: {len(data)}")

first_key = next(iter(data))

print("Sample Key:")
print(first_key)

print("\nSample Content:")
print(json.dumps(data[first_key], indent=4))

Total samples: 1029726
Sample Key:
track_crash_in_library_outdoor/8e7edea3da.jpg

Sample Content:
{
    "url": "https://images-na.ssl-images-amazon.com/images/I/91ywrXSiLPL._SY355_.jpg",
    "incidents": {
        "truck accident": 0
    },
    "places": {}
}


In [20]:
# ==========================================================
# Cell 4.1 - Common Utility Functions
# ==========================================================
import gc
import random
import requests
import numpy as np
import torch
from io import BytesIO
from PIL import Image

# ----------------------------------------------------------
# GPU Memory Cleanup
# ----------------------------------------------------------
def cleanup():
    """
    Release CPU/GPU memory.
    """
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
# ----------------------------------------------------------
# Set Random Seed
# ----------------------------------------------------------

def set_seed(seed: int = 42):
    """
    Set random seed for reproducibility.
    """ 
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
# ----------------------------------------------------------
# Download Image
# ----------------------------------------------------------
def download_image(
    url: str,
    headers=None,
    timeout: int = 30,
    retries: int = 3,
):
    """
    Download image from URL.
    Returns
    -------
    bytes | None
    """
    for attempt in range(retries):
        try:
            response = requests.get(
                url,
                headers=headers,
                timeout=timeout,
            )
            response.raise_for_status()

            return response.content

        except Exception as e:
            print(
                f"[Retry {attempt+1}/{retries}] {e}"
            )
    return None

# ----------------------------------------------------------
# Resize + Center Crop
# ----------------------------------------------------------

def resize_center_crop(
    image: Image.Image,
    size: int = 512,
):
    """
    Resize while preserving aspect ratio,
    then center crop to (size x size).
    """
    width, height = image.size
    scale = max(
        size / width,
        size / height,
    )
    new_width = int(width * scale)
    new_height = int(height * scale)
    image = image.resize(
        (new_width, new_height),
        Image.LANCZOS,
    )
    left = (new_width - size) // 2
    top = (new_height - size) // 2
    image = image.crop(
        (
            left,
            top,
            left + size,
            top + size,
        )
    )
    return image
# ----------------------------------------------------------
# Save Generated Image
# ----------------------------------------------------------
def save_generated_image(
    image: Image.Image,
    output_dir: str,
    image_name: str,
):
    """
    Save generated image.
    """
    os.makedirs(
        output_dir,
        exist_ok=True,
    )
    output_path = os.path.join(
        output_dir,
        image_name,
    )
    image.save(output_path)

    return output_path
# ----------------------------------------------------------
# Print GPU Memory
# ----------------------------------------------------------
def print_gpu_memory():
    if not torch.cuda.is_available():
        print("CUDA is not available.")
        return
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"Allocated : {allocated:.2f} GB")
    print(f"Reserved  : {reserved:.2f} GB")

In [10]:
# ==========================================================
# Cell 5 - Loading Gemma Model 
# ==========================================================
from transformers import AutoProcessor, Gemma3ForConditionalGeneration
import torch

model_id = GENERAL_MODEL 

# AutoModelForImageTextToText.from_pretrainted - Neu dung gemma4 

vision_model = Gemma3ForConditionalGeneration.from_pretrained(
    model_id, torch_dtype=torch.bfloat16, device_map="auto", token=hf_token
).eval()

vision_processor = AutoProcessor.from_pretrained(model_id, token=hf_token)

print(f"Loading model {model_id} successfully")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Loading model google/gemma-3-4b-it successfully


In [32]:
# ==========================================================
# Cell 6 - Vision Model Understanding (Gemma 3 - 4 it)
# ==========================================================
def generate_scene_description(
    image
): 
    """
        Analyze the image and generate the image description about the scene 
    
    """
    prompt = """
            You are a professional disaster scene analysis assistant.
        Your task is ONLY to describe what is directly visible in the image.
        Rules:
        - Describe only visible objects.
        - Do not infer hidden information.
        - Do not speculate.
        - Do not explain the cause of the disaster.
        - Do not suggest rescue actions.
        - Do not mention anything not visible.
        - Return a single factual paragraph.
    """
    messages = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text", 
                    "text": "You are a Vision-Language AI assistant specialized in disaster scene understanding and image editing instruction generation."}
            ]
        },
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt}
            ]
        }
    ]
    # Tu cai document cua hugging face nhe may em :v 
    inputs = vision_processor.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors="pt").to(vision_model.device)
    
    input_len = inputs["input_ids"].shape[-1]
    # Gemma 3 
    with torch.inference_mode():
        generation = vision_model.generate(**inputs, max_new_tokens=100, do_sample=False)
        generation = generation[0][input_len:]
    
    decoded = vision_processor.decode(generation, skip_special_tokens=True).strip()
    # Gemma 3 

    ### Gemma 4 
    """
    with torch.inference_mode(): 
        outputs = vision_model.generate(**inputs, max_new_tokens=1024) 
        generation = outputs[0][input_len:]
    response = vision_processor.decode(generation, skip_special_tokens=False)
    return vision_processor.parse_response(response)
    """
    return decoded 

In [13]:
# No Long use this cell 
# ==========================================================
# Load Large Language Model (4-bit)
# ==========================================================

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)


def load_llm():
    """
    Load Qwen2.5-7B-Instruct using 4-bit quantization.
    """

    print("=" * 60)
    print("Loading Large Language Model...")
    print("=" * 60)

    tokenizer = AutoTokenizer.from_pretrained(
        LLM_MODEL,
        token=HF_TOKEN,
        trust_remote_code=True,
    )

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        token=HF_TOKEN,
        trust_remote_code=True,
    )

    model.eval()

    print("=" * 60)
    print("LLM Loaded Successfully")
    print("=" * 60)

    return tokenizer, model

In [14]:
# No longer use this cell 
# Cell 7.1 - Load LLM 
cleanup()
print("=" * 80)
print("Loading LLM...")
print("=" * 80)
tokenizer, llm_model = load_llm()
print("=" * 80)
print("LLM Ready")
print("=" * 80)

Loading LLM...
Loading Large Language Model...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

LLM Loaded Successfully
LLM Ready


In [35]:
# ==========================================================
# Cell 8 - Generate Rescue Instructions
# ==========================================================

import torch


def generate_rescue_instruction(
    scene_description: str
) -> str:
    """
    Generate rescue image editing instructions from
    the disaster scene description.
    """

    system_prompt = """
        You are an expert emergency rescue planner.
        
        Your task is to generate editing instructions for an image editing model.
        Requirements:
        1. Preserve the original disaster scene.
        2. Preserve damaged buildings and existing objects.
        3. Do not change the disaster type.
        4. Add only realistic rescue operations.
        5. Add rescue personnel when appropriate.
        6. Add rescue vehicles when appropriate.
        7. Add emergency equipment when appropriate.
        8. Maintain realistic object scale.
        9. Maintain realistic lighting.
        10. Maintain realistic perspective.
        11. Keep all newly added objects consistent with the existing environment.
        
        Return ONLY the editing instructions.
        Do not explain your reasoning.
        Do not describe the original image.
        Do not use markdown.
    """
    user_prompt = f"""
        Disaster Scene:
        {scene_description}
        Generate image editing instructions.
    """

    messages = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": system_prompt,
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": user_prompt,
                }
            ],
        },
    ]

    # Gemma 3
    inputs = vision_processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to(vision_model.device)
    
    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        generation = vision_model.generate(**inputs, max_new_tokens=320, do_sample=False)
    generation = generation[0][input_len:]
    instruction = vision_processor.decode(
        generation,
        skip_special_tokens=True,
    ).strip()
    return instruction
    # Gemma 3 
    ### Gemma 4 
    """
    with torch.inference_mode(): 
        outputs = vision_model.generate(**inputs, max_new_tokens=1024, do_sample=False)
    generation = outputs[0][input_len:]
    instruction = vision_processor.decode(generation, skip_special_tokens=False).strip()
    return vision_processor.parse_response(instruction)
    """
    ###

In [14]:
# ==========================================================
# Cell 9 - Build FLUX Prompt
# ==========================================================
def build_flux_prompt(
    scene_description: str,
    rescue_instruction: str,
) -> str:
    """
    Build the final prompt for FLUX image editing.
    """
    prompt = f"""
You are editing an existing disaster photograph.

Original Scene
--------------
{scene_description}
Editing Instructions
--------------------
{rescue_instruction}
Requirements
    - Preserve the original disaster scene.
    - Preserve all existing buildings, vehicles, roads and environmental objects.
    - Do not change the disaster type.
    - Add only realistic rescue operations.
    - Blend newly added rescue personnel, vehicles and equipment naturally.
    - Maintain realistic lighting, shadows and perspective.
    - Maintain correct object proportions.
    - Generate anatomically correct humans.
    - Produce seamless image editing without visible artifacts.
    Style
    - Documentary disaster photography
    - Photojournalism
    - Real-world emergency response
    - Natural color grading
    - Authentic textures
    - High realism
    - Non-cinematic
"""
    return prompt.strip()

In [15]:
# ==========================================================
# Cell 10 - Load FLUX2-klein-4B
# ==========================================================
import torch
from diffusers import Flux2KleinPipeline
def load_flux_model():
    """
    Load FLUX2-klein-4B once.
    """
    print("=" * 60)
    print("Loading FLUX2-klein-4B...")
    print("=" * 60)
    pipe = Flux2KleinPipeline.from_pretrained(
        FLUX_MODEL,
        torch_dtype=torch.float16,
        # Neu chi co 1 GPU thi bo phan device_map, max_memory 
        device_map="balanced",
        max_memory={
            0: "13GiB",
            1: "7GiB",
        },
        token=HF_TOKEN,
    )
    # Chi co 1 GPU: pipe.to("cuda")
    try:
        pipe.enable_attention_slicing()
    except Exception:
        pass
    pipe.set_progress_bar_config(
        disable=False,
    )
    print("=" * 60)
    print("FLUX Loaded Successfully")
    print("=" * 60)
    return pipe
pipe = load_flux_model() 

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Loading FLUX2-klein-4B...


model_index.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

FLUX Loaded Successfully


In [17]:
# ==========================================================
# Cell 11 - Image Generation
# ==========================================================

import torch
from PIL import Image
def generate_rescue_image(
    pipe,
    image: Image.Image,
    prompt: str,
    guidance_scale: float = 4.0,
    num_inference_steps: int = 30,
    seed: int = 42,
):
    """
    Generate a rescue simulation image using FLUX2-klein-4B.
    """
    generator = torch.Generator(
        device="cpu"
    ).manual_seed(seed)
    with torch.inference_mode():

        result = pipe(
            image=image,
            prompt=prompt,
            guidance_scale=guidance_scale,
            num_inference_steps=num_inference_steps,
            generator=generator,
            output_type="pil",
        )
    generated_image = result.images[0]
    del result
    del generator
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return generated_image

In [36]:
# ==========================================================
# Cell 12 - Main Pipeline
# ==========================================================

import os
import re
import json
import traceback
from io import BytesIO

count = 0
limiter_removal = 4
skipped = 0
evaluation_records = []
metadata_dir = os.path.join(
    OUTPUT_DIR,
    "_metadata",
)

os.makedirs(
    metadata_dir,
    exist_ok=True,
)

print("=" * 80)
print("Pipeline Started")
print("=" * 80)

for img_key, img_info in data.items():

    if count >= LIMIT_IMAGES:
        break

    print("=" * 80)
    print(f"Processing: {img_key}")
    print("=" * 80)

    try:

        # ==================================================
        # 1. Check Positive Labels
        # ==================================================

        incidents = img_info.get(
            "incidents",
            {}
        )

        positive_incidents = [
            incident
            for incident, value in incidents.items()
            if value == 1
        ]

        if not positive_incidents:
            print(
                f"Skip: {img_key} (No Positive Labels)"
            )

            skipped += 1
            continue

        # ==================================================
        # 2. Resume Check
        # ==================================================

        safe_name = re.sub(
            r"[^a-zA-Z0-9]+",
            "_",
            os.path.splitext(
                os.path.basename(img_key)
            )[0],
        ).strip("_")

        output_path = os.path.join(
            OUTPUT_DIR,
            f"{safe_name}.png",
        )

        if os.path.exists(output_path):

            print(
                f"Skip: {img_key} (Already Generated)"
            )

            skipped += 1
            continue

        # ==================================================
        # 3. Download Image
        # ==================================================

        url = img_info.get("url")

        if not url:

            print("Skip: Missing URL")

            skipped += 1
            continue

        content = download_image(
            url,
            headers=HEADERS,
            timeout=REQUEST_TIMEOUT,
            retries=DOWNLOAD_RETRIES,
        )
        if content is None:
            print("Skip: Download Failed")
            skipped += 1
            continue
        try:
            original_image = Image.open(
                BytesIO(content)
            ).convert("RGB")

        except Exception as e:

            print(
                f"Skip Invalid Image: {e}"
            )
            skipped += 1
            continue
        original_image = resize_center_crop(
            original_image,
            IMAGE_SIZE,
        )
        if torch.cuda.is_available():
            print(
                "GPU Memory:",
                round(
                    torch.cuda.memory_allocated()
                    / 1024 ** 3,
                    2,
                ),
                "GB",
            )
        # ==================================================
        # 4. Vision Understanding
        # ==================================================

        try:
            scene_description = generate_scene_description(
                image=original_image
            )
        except Exception as e:
            print(
                f"Generate Image Error: {e}"
            )
            skipped += 1
            del original_image
            cleanup()
            continue

        print("\nScene Description")
        print("-" * 60)
        print(scene_description)
        # ==================================================
        # 5. Generate Rescue Instruction
        # ==================================================
        try:

            rescue_instruction = generate_rescue_instruction(
                scene_description=scene_description
            )
        except Exception as e:

            print(
                f"LLM Error: {e}"
            )
            skipped += 1
            del original_image
            cleanup()
            continue
        print("\nEditing Instruction")
        print("-" * 60)
        print(rescue_instruction)
        # ==================================================
        # 6. Build FLUX Prompt
        # ==================================================
        flux_prompt = build_flux_prompt(
            scene_description,
            rescue_instruction,
        )
        print("\nFLUX Prompt")
        print("-" * 60)
        print(flux_prompt)
        # ==================================================
        # 7. Generate Rescue Image
        # ==================================================
        try:
            generated_image = generate_rescue_image(
                pipe=pipe,
                image=original_image,
                prompt=flux_prompt,
                guidance_scale=GUIDANCE_SCALE,
                num_inference_steps=5,
                seed=42 + count,
            )

        except torch.cuda.OutOfMemoryError:

            print(
                "FLUX CUDA Out Of Memory"
            )
            skipped += 1
            del original_image
            cleanup()
            continue
        # ==================================================
        # 8. Save Image
        # ==================================================
        generated_image.save(
            output_path,
        )
        print(
            f"Saved -> {output_path}"
        )
        # ==================================================
        # 9. Save Metadata
        # ==================================================

        metadata = {
            "image_key": img_key,
            "labels": positive_incidents,
            "url": url,
            "scene_description": scene_description,
            "editing_instruction": rescue_instruction,
            "flux_prompt": flux_prompt,
            "output_path": output_path,
        }
        evaluation_records.append(metadata)
        with open(
            os.path.join(
                metadata_dir,
                f"{safe_name}.json",
            ),
            "w",
            encoding="utf-8",
        ) as f:

            json.dump(
                metadata,
                f,
                indent=2,
                ensure_ascii=False,
            )

        count += 1

        print(
            f"Progress: {count}/{LIMIT_IMAGES}"
        )
        # ==================================================
        # 10. Cleanup
        # ==================================================
        del original_image
        del generated_image
        del scene_description
        del rescue_instruction
        del flux_prompt
        cleanup()
    except Exception:
        print("\nUnexpected Error")
        traceback.print_exc()
        cleanup()
        skipped += 1
        continue
print("\n" + "=" * 80)
print("Finished")
print(f"Generated : {count}")
print(f"Skipped   : {skipped}")
print(f"Output    : {OUTPUT_DIR}")

Pipeline Started
Processing: track_crash_in_library_outdoor/8e7edea3da.jpg
Skip: track_crash_in_library_outdoor/8e7edea3da.jpg (No Positive Labels)
Processing: van_wreck_in_ski_slope/816a13d3c4.jpg
Skip: van_wreck_in_ski_slope/816a13d3c4.jpg (No Positive Labels)
Processing: motorcycle_incident_in_mosque_outdoor/114adb9c2c.jpg
Skip: motorcycle_incident_in_mosque_outdoor/114adb9c2c.jpg (No Positive Labels)
Processing: burned_motel/09d4d7e4b5.jpg
Skip: burned_motel/09d4d7e4b5.jpg (No Positive Labels)
Processing: boat_crash_in_landfill/f064995437.jpg
Skip: boat_crash_in_landfill/f064995437.jpg (No Positive Labels)
Processing: van_incident_in_dam/1c902ee1cc.jpg
Skip: van_incident_in_dam/1c902ee1cc.jpg (No Positive Labels)
Processing: drought_in_police_station/3aa52361fb.jpg
Skip: drought_in_police_station/3aa52361fb.jpg (No Positive Labels)
Processing: van_accident_in_street/00323.jpg
[Retry 1/3] 404 Client Error: Not Found for url: https://www.khou.com/img/resize/content.khou.com/photo/201

  0%|          | 0/5 [00:00<?, ?it/s]

Saved -> /kaggle/working/Rescue_Image_Generation/00188.png
Progress: 1/5
Processing: car_accident_in_inn_outdoor/00124.jpg
GPU Memory: 11.5 GB

Scene Description
------------------------------------------------------------
The image shows a damaged black sedan with its hood up and front end crumpled. A large white semi-truck is partially visible behind the sedan, with damage to its trailer. A construction worker wearing an orange vest, a white hard hat, and jeans is walking away from the scene, carrying a white bag. Several emergency personnel are standing near the truck, and traffic is backed up on the highway. A green sign indicating “Green Mountain” is visible in the background, and a power pole is also

Editing Instruction
------------------------------------------------------------
Enhance the scene with a triage tent being erected near the damaged vehicles. Add paramedics attending to the construction worker, administering first aid. Introduce a fire truck arriving at the scene, 

  0%|          | 0/5 [00:00<?, ?it/s]

Saved -> /kaggle/working/Rescue_Image_Generation/00124.png
Progress: 2/5
Processing: collapsed_sky/00317.jpg
GPU Memory: 11.5 GB

Scene Description
------------------------------------------------------------
The image shows a large, dark building that has collapsed and is leaning at a significant angle. Several heavy machinery vehicles, including excavators and a Volvo, are present at the site of the collapse. There are firefighters in orange suits visible near the debris. In the background, there are residential buildings, a bridge, and a hilly landscape with trees.

Editing Instruction
------------------------------------------------------------
Enhance the scene with a heavy-duty crane positioned near the collapsed building, carefully lifting debris. Add a medical tent with several paramedics attending to injured individuals near the crane. Include a water truck providing hydration to rescue personnel. Introduce a team of search and rescue dogs working alongside firefighters, sniff

...

Saved -> /kaggle/working/Rescue_Image_Generation/00317.png
Progress: 3/5
Processing: under_construction_junkyard/6a77d2e347.jpg
Skip: under_construction_junkyard/6a77d2e347.jpg (No Positive Labels)
Processing: plane_crash_in_volcano/dd5603ad16.jpg
Skip: plane_crash_in_volcano/dd5603ad16.jpg (No Positive Labels)
Processing: destroyed_lake_natural/00291.jpg
[Retry 1/3] 404 Client Error: Not Found for url: https://revitalization.org/wp-content/uploads/2019/11/Upper-Truckee-Marsh-aerial_June2017_1400.jpg
[Retry 2/3] 404 Client Error: Not Found for url: https://revitalization.org/wp-content/uploads/2019/11/Upper-Truckee-Marsh-aerial_June2017_1400.jpg
[Retry 3/3] 404 Client Error: Not Found for url: https://revitalization.org/wp-content/uploads/2019/11/Upper-Truckee-Marsh-aerial_June2017_1400.jpg
Skip: Download Failed
Processing: firestorm_in_highway/8ea17d4b88.jpg
GPU Memory: 11.5 GB

Scene Description
------------------------------------------------------------
The image shows a highway wi

  0%|          | 0/5 [00:00<?, ?it/s]

Saved -> /kaggle/working/Rescue_Image_Generation/8ea17d4b88.png
Progress: 4/5
Processing: train_accident_in_embassy/3ae3476a88.jpg
GPU Memory: 11.5 GB

Scene Description
------------------------------------------------------------
The image shows a damaged white bus with the words “BABT” on the side, lying on its side in a gravel area. A dark-colored tank car is partially visible behind the bus. A person wearing a yellow vest is visible inside the bus. Power lines are running above the scene, and there are trees and foliage in the background under a blue sky with white clouds.

Editing Instruction
------------------------------------------------------------
Enhance the scene with a triage tent and medical personnel attending to the occupant of the bus. Add a fire truck with flashing lights positioned nearby, focused on assessing the bus for fire hazards. Include a heavy-duty rescue vehicle, such as a wrecker, attempting to upright the bus. Introduce a portable generator providing power

  0%|          | 0/5 [00:00<?, ?it/s]

Saved -> /kaggle/working/Rescue_Image_Generation/3ae3476a88.png
Progress: 5/5

Finished
Generated : 5
Skipped   : 11
Output    : /kaggle/working/Rescue_Image_Generation


In [ ]:
# ==========================================================
# Cell 13 - Export Evaluation Report
# ==========================================================

import pandas as pd
import os

print("=" * 80)
print("Export Evaluation Report")
print("=" * 80)

if len(evaluation_records) == 0:

    print("No evaluation records.")

else:

    df = pd.DataFrame(
        evaluation_records
    )

    csv_path = os.path.join(
        OUTPUT_DIR,
        "evaluation_report.csv",
    )

    df.to_csv(
        csv_path,
        index=False,
    )

    print(
        f"CSV Report : {csv_path}"
    )

    metric_columns = [
        col
        for col in [
            "psnr",
            "ssim",
            "clip_score",
        ]
        if col in df.columns
    ]
    if metric_columns:
        print()
        print(
            df[metric_columns].describe()
        )

print("=" * 80)
print("Finished")
print("=" * 80)